# Homework 2 - Training and Optimization

In [12]:
import tensorflow as tf
import numpy as np
import os
import zipfile
from glob import glob
from reader import AudioReader
from preprocessing import Padding, Normalization
from preprocessing import MelSpectrogram, MFCC

In [13]:
# Environment variables
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

### Define the hyperparameters

In [14]:
SCRIPT_DIR = os.path.abspath('')

PREPROCESSING_ARGS = {
    'sampling_rate': 16000,
    'frame_length_in_s': 0.008,
    'frame_step_in_s': 0.002,
    'num_mel_bins': 50,
    'lower_frequency': 30,
    'upper_frequency': 6000,
    'num_coefficients': 0
}

TRAINING_ARGS = {
    'batch_size': 20,
    'learning_rate': 1.e-2,
    'end_learning_rate': 1.e-4,
    'epochs': 40,
    'width_multiplier': [0.25, 0.5, 0.75], # structured pruning
}

LABELS = ['down', 'up']

### Create train/val/test Datasets and Callback

In [15]:
train_ds = tf.data.Dataset.list_files([os.path.join(SCRIPT_DIR, 'msc-train/down*'), os.path.join(SCRIPT_DIR, 'msc-train/up*')])
val_ds = tf.data.Dataset.list_files([os.path.join(SCRIPT_DIR, 'msc-val/down*'), os.path.join(SCRIPT_DIR, 'msc-val/up*')])
test_ds = tf.data.Dataset.list_files([os.path.join(SCRIPT_DIR, 'msc-test/down*'), os.path.join(SCRIPT_DIR, 'msc-test/up*')])

In [16]:
# Learning Rate scheduler
linear_decay = tf.keras.optimizers.schedules.PolynomialDecay(
    initial_learning_rate=TRAINING_ARGS['learning_rate'],
    end_learning_rate=TRAINING_ARGS['end_learning_rate'],
    decay_steps=int(tf.data.experimental.cardinality(train_ds)/TRAINING_ARGS['batch_size']) * TRAINING_ARGS['epochs'],
    power = 1.0
)

exponential_decay = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=TRAINING_ARGS['learning_rate'],
    decay_steps=int(tf.data.experimental.cardinality(train_ds)/TRAINING_ARGS['batch_size']),
    decay_rate = (TRAINING_ARGS['end_learning_rate'] / TRAINING_ARGS['learning_rate'])**(1/30),
    staircase = True
)
lr_scheduler = tf.keras.callbacks.LearningRateScheduler(exponential_decay)

# Early Stopping
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=8,
    verbose=1,
    mode='min'
)

### Define the data pipeline

In [17]:
audio_reader = AudioReader(tf.int16)
padding = Padding(PREPROCESSING_ARGS['sampling_rate'])
normalization = Normalization(tf.int16)

if PREPROCESSING_ARGS['num_coefficients'] == 0:
    PREPROCESSING_ARGS.pop('num_coefficients')
    feature_processor = MelSpectrogram(**PREPROCESSING_ARGS)
    feature_processor_fn = feature_processor.get_mel_spec
    feature_processor_fn_lab = feature_processor.get_mel_spec_and_label
else:
    feature_processor = MFCC(**PREPROCESSING_ARGS)
    feature_processor_fn = feature_processor.get_mfccs
    feature_processor_fn_lab = feature_processor.get_mfccs_and_label

LABELS = ['down', 'up']

def prepare_for_training(feature, label):
    feature = tf.expand_dims(feature, -1)
    label_id = tf.argmax(label == LABELS)

    return feature, label_id

train_ds = (train_ds
            .map(audio_reader.get_audio_and_label)
            .map(padding.pad)
            .map(normalization.normalize)
            .map(feature_processor_fn_lab)
            .map(prepare_for_training)
            .batch(TRAINING_ARGS['batch_size'])
            .cache())
val_ds = (val_ds
            .map(audio_reader.get_audio_and_label)
            .map(padding.pad)
            .map(normalization.normalize)
            .map(feature_processor_fn_lab)
            .map(prepare_for_training)
            .batch(TRAINING_ARGS['batch_size']))
test_ds = (test_ds
            .map(audio_reader.get_audio_and_label)
            .map(padding.pad)
            .map(normalization.normalize)
            .map(feature_processor_fn_lab)
            .map(prepare_for_training)
            .batch(TRAINING_ARGS['batch_size']))

### Define the data pipeline 

### Read a batch of data

In [18]:
for example_batch, example_labels in train_ds.take(1):
  print('Batch Shape:', example_batch.shape)
  print('Data Shape:', example_batch.shape[1:])
  print('Labels:', example_labels)
     

Batch Shape: (20, 497, 50, 1)
Data Shape: (497, 50, 1)
Labels: tf.Tensor([1 0 0 1 1 1 0 0 1 1 1 0 1 1 1 1 0 0 0 0], shape=(20,), dtype=int64)


2024-12-17 21:05:49.192426: W tensorflow/core/kernels/data/cache_dataset_ops.cc:858] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


### Create the model

In [19]:
wm = TRAINING_ARGS['width_multiplier']

model_1 = tf.keras.Sequential([
    tf.keras.layers.Input(shape=example_batch.shape[1:]),
    tf.keras.layers.Conv2D(filters=128, kernel_size=[3, 3], strides=[2, 2], use_bias=False, padding='valid'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),
    tf.keras.layers.Conv2D(filters=128, kernel_size=[3, 3], strides=[1, 1], use_bias=False, padding='same'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),
    tf.keras.layers.Conv2D(filters=128, kernel_size=[3, 3], strides=[1, 1], use_bias=False, padding='same'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(units=len(LABELS)),
    tf.keras.layers.Softmax()
]) # originally 128 filters and 3 conv2d layers

In [20]:
model_1.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_3 (Conv2D)           (None, 248, 24, 128)      1152      
                                                                 
 batch_normalization_3 (Bat  (None, 248, 24, 128)      512       
 chNormalization)                                                
                                                                 
 re_lu_3 (ReLU)              (None, 248, 24, 128)      0         
                                                                 
 conv2d_4 (Conv2D)           (None, 248, 24, 128)      147456    
                                                                 
 batch_normalization_4 (Bat  (None, 248, 24, 128)      512       
 chNormalization)                                                
                                                                 
 re_lu_4 (ReLU)              (None, 248, 24, 128)     

### Train the model with callback

In [21]:
loss = tf.losses.SparseCategoricalCrossentropy(from_logits=False)
optimizer = tf.optimizers.Adam(learning_rate=exponential_decay)
metrics = [tf.metrics.SparseCategoricalAccuracy()]
model_1.compile(loss=loss, optimizer=optimizer, metrics=metrics)

history = model_1.fit(
    train_ds, 
    epochs=TRAINING_ARGS['epochs'], 
    validation_data=val_ds, 
    callbacks=[lr_scheduler, early_stopping]
)

Epoch 1/40


2024-12-17 21:05:52.283726: E external/local_xla/xla/stream_executor/gpu/gpu_cudamallocasync_allocator.cc:306] gpu_async_0 cuMemAllocAsync failed to allocate 457965568 bytes: CUDA error: out of memory (CUDA_ERROR_OUT_OF_MEMORY)
 Reported by CUDA: Free memory/Total memory: 113180672/3899326464
2024-12-17 21:05:52.283769: E external/local_xla/xla/stream_executor/gpu/gpu_cudamallocasync_allocator.cc:311] Stats: Limit:                       774832128
InUse:                       514113294
MaxInUse:                    881858562
NumAllocs:                       45898
MaxAllocSize:                565313536
Reserved:                            0
PeakReserved:                        0
LargestFreeBlock:                    0

2024-12-17 21:05:52.283783: E external/local_xla/xla/stream_executor/gpu/gpu_cudamallocasync_allocator.cc:63] Histogram of current allocation: (allocation_size_in_bytes, nb_allocation_of_that_sizes), ...;
2024-12-17 21:05:52.283790: E external/local_xla/xla/stream_executor/g

80/80 [==============================] - 13s 105ms/step - loss: 0.6205 - sparse_categorical_accuracy: 0.7006 - val_loss: 0.8169 - val_sparse_categorical_accuracy: 0.5550 - lr: 0.0100
Epoch 2/40
80/80 [==============================] - 8s 99ms/step - loss: 0.5590 - sparse_categorical_accuracy: 0.7387 - val_loss: 0.8314 - val_sparse_categorical_accuracy: 0.6000 - lr: 0.0086
Epoch 3/40
80/80 [==============================] - 8s 99ms/step - loss: 0.5349 - sparse_categorical_accuracy: 0.7594 - val_loss: 0.5823 - val_sparse_categorical_accuracy: 0.7350 - lr: 0.0074
Epoch 4/40
80/80 [==============================] - 8s 98ms/step - loss: 0.5243 - sparse_categorical_accuracy: 0.7725 - val_loss: 0.9610 - val_sparse_categorical_accuracy: 0.5250 - lr: 0.0063
Epoch 5/40
80/80 [==============================] - 8s 99ms/step - loss: 0.5172 - sparse_categorical_accuracy: 0.7844 - val_loss: 0.7391 - val_sparse_categorical_accuracy: 0.5900 - lr: 0.0054
Epoch 6/40
80/80 [==============================]

### Show the history

In [22]:
history.history

{'loss': [0.6204770803451538,
  0.559004008769989,
  0.5348823666572571,
  0.524314820766449,
  0.5172159671783447,
  0.5042198896408081,
  0.49164000153541565,
  0.4808470904827118,
  0.4706399440765381,
  0.4630095958709717,
  0.4568292200565338,
  0.45041629672050476,
  0.44431963562965393,
  0.4392797350883484,
  0.4349205791950226,
  0.4311122000217438,
  0.42805615067481995,
  0.4254385232925415,
  0.42327308654785156,
  0.4212014079093933,
  0.4192735254764557,
  0.41740497946739197,
  0.41573575139045715,
  0.4141543507575989,
  0.4126885235309601,
  0.4112085700035095,
  0.4097820520401001,
  0.40839335322380066,
  0.40710097551345825,
  0.4059090316295624,
  0.40483415126800537,
  0.40388134121894836,
  0.40303635597229004,
  0.4022909998893738,
  0.4016226530075073,
  0.40103432536125183,
  0.40052497386932373,
  0.4000917077064514,
  0.3997249901294708],
 'sparse_categorical_accuracy': [0.7006250023841858,
  0.7387499809265137,
  0.7593749761581421,
  0.7724999785423279,
  

### Evaluate the model

In [23]:
training_loss = history.history['loss'][-1]
training_accuracy = history.history['sparse_categorical_accuracy'][-1]
val_loss = history.history['val_loss'][-1]
val_accuracy = history.history['val_sparse_categorical_accuracy'][-1]

test_loss, test_accuracy = model_1.evaluate(test_ds)

print(f'Training Loss: {training_loss:.4f}')
print(f'Training Accuracy: {training_accuracy*100.:.2f}%')
print()
print(f'Validation Loss: {val_loss:.4f}')
print(f'Validation Accuracy: {val_accuracy*100.:.2f}%')
print()
print(f'Test Loss: {test_loss:.4f}')
print(f'Test Accuracy: {test_accuracy*100.:.2f}%')

10/10 [==============================] - 1s 32ms/step - loss: 0.4029 - sparse_categorical_accuracy: 0.8700
Training Loss: 0.3997
Training Accuracy: 87.75%

Validation Loss: 0.4451
Validation Accuracy: 84.50%

Test Loss: 0.4029
Test Accuracy: 87.00%


### Save hyperparameters & results

In [24]:
import os
from time import time

timestamp = int(time())

saved_model_dir = f'./saved_models/{timestamp}'
if not os.path.exists(saved_model_dir):
    os.makedirs(saved_model_dir)
model_1.save(saved_model_dir)

INFO:tensorflow:Assets written to: ./saved_models/1734466267/assets


INFO:tensorflow:Assets written to: ./saved_models/1734466267/assets


In [25]:
import pandas as pd

output_dict = {
    'timestamp': timestamp,
    **PREPROCESSING_ARGS,
    **TRAINING_ARGS,
    'test_accuracy': test_accuracy
}

df = pd.DataFrame([output_dict])

output_path='./mel_spectrogram_results.csv'
df.to_csv(output_path, mode='a', header=not os.path.exists(output_path), index=False)

### TFLite Conversion

In [26]:
# Converting a SavedModel to a TensorFlow Lite model.
converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
tflite_model = converter.convert()

tflite_model_dir = './tflite_models'
# create the path if it doesn't exist
if not os.path.exists(tflite_model_dir):
    os.makedirs(tflite_model_dir)

# name the model
tflite_model_name = os.path.join(tflite_model_dir, f'{timestamp}.tflite')
tflite_model_name

# write the model
with open(tflite_model_name, 'wb') as fp:
    fp.write(tflite_model)

2024-12-17 21:11:09.482143: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2024-12-17 21:11:09.482178: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2024-12-17 21:11:09.482579: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: ./saved_models/1734466267
2024-12-17 21:11:09.486026: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2024-12-17 21:11:09.486051: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: ./saved_models/1734466267
2024-12-17 21:11:09.493002: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:388] MLIR V1 optimization pass is not enabled
2024-12-17 21:11:09.495737: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2024-12-17 21:11:09.569225: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: ./save

### Evaluate TFLite Model

In [27]:
MODEL_FILE_PATH = tflite_model_name

# Measure model size
model_size = os.path.getsize(MODEL_FILE_PATH)

if MODEL_FILE_PATH.endswith('.zip'):
    with zipfile.ZipFile(MODEL_FILE_PATH, 'r') as fp:
        fp.extractall('/tmp/')
        model_filename = fp.namelist()[0]
        MODEL_FILE_PATH = '/tmp/' + model_filename

interpreter = tf.lite.Interpreter(model_path=MODEL_FILE_PATH)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Number of inputs:", len(input_details))
print("Number of outputs:", len(output_details))
print("Input name:", input_details[0]['name'])
print("Input shape:", input_details[0]['shape'])
print("Output name:", output_details[0]['name'])
print("Output shape:", output_details[0]['shape'])

Number of inputs: 1
Number of outputs: 1
Input name: serving_default_input_2:0
Input shape: [  1 497  50   1]
Output name: StatefulPartitionedCall:0
Output shape: [1 2]


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [28]:
SCRIPT_DIR = os.path.abspath('')

filenames = glob(os.path.join(SCRIPT_DIR, 'msc-test/down*')) + glob(os.path.join(SCRIPT_DIR, 'msc-test/up*'))

accuracy = 0.0

for filename in filenames:
    audio, true_label = audio_reader.get_audio_and_label(filename)   
    true_label = true_label.numpy().decode()
    
    audio = padding.pad_audio(audio)
    audio = normalization.normalize_audio(audio)
    features = feature_processor_fn(audio)
    features = tf.expand_dims(features, 0)
    features = tf.expand_dims(features, -1)

    interpreter.set_tensor(input_details[0]['index'], features)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details[0]['index'])

    top_index = np.argmax(output[0])
    predicted_label = LABELS[top_index]

    accuracy += true_label == predicted_label

accuracy /= len(filenames)

In [29]:
print(f'Accuracy: {100 * accuracy:.3f}%')
print(f'Model size: {model_size / 2 ** 10:.1f}KB')

Accuracy: 87.000%
Model size: 1162.1KB


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=b4ef5aa4-3f71-4837-91f1-c6fd9810a7ea' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>